In [1]:
import pandas as pd
import numpy as np

### **UCDP**
---

UCDP Georeferenced Event Dataset (GED) Global version 25.1 spanning from 1989-01 to 2024-12

In [7]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv')
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence', 'region', 'conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';')
df_ucdp = df_ucdp.merge(isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left').rename(columns = {'alpha_3':'isocode'})
df_ucdp

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/1082166062.py:1: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv')


,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,region,conflict_name,side_a,side_b,name,isocode
0,Afghanistan,259,2017-07-31 00:00:00.000,2017-07-31 00:00:00.000,6,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
1,Afghanistan,259,2021-08-26 00:00:00.000,2021-08-26 00:00:00.000,183,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
2,Afghanistan,259,2021-08-28 00:00:00.000,2021-08-28 00:00:00.000,2,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
3,Afghanistan,259,2021-08-29 00:00:00.000,2021-08-29 00:00:00.000,10,524,1,Asia,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG
4,Afghanistan,333,1989-01-01 00:00:00.000,1989-01-01 00:00:00.000,4,727,1,Asia,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
385913,Zimbabwe (Rhodesia),562,1989-03-18 00:00:00.000,1989-03-24 00:00:00.000,9,1029,3,Africa,Renamo - Civilians,Renamo,Civilians,Zimbabwe (Rhodesia),ZWE
385914,Zimbabwe (Rhodesia),562,1989-07-09 00:00:00.000,1989-07-15 00:00:00.000,9,1029,3,Africa,Renamo - Civilians,Renamo,Civilians,Zimbabwe (Rhodesia),ZWE
385915,Zimbabwe (Rhodesia),562,1990-06-07 00:00:00.000,1990-06-10 00:00:00.000,7,1029,3,Africa,Renamo - Civilians,Renamo,Civilians,Zimbabwe (Rhodesia),ZWE
385916,Zimbabwe (Rhodesia),562,1990-11-17 00:00:00.000,1990-11-23 00:00:00.000,1,1029,3,Africa,Renamo - Civilians,Renamo,Civilians,Zimbabwe (Rhodesia),ZWE


In [8]:
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"])
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"])

df_ucdp["date_start"] = df_ucdp["date_start"].dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = df_ucdp["date_end"].dt.to_period("M").dt.to_timestamp()


#1. Expand the events that cross multiple months and distribute 'best' value evenly
#------------------------------------------------------------------------------------
expanded_events = []

for _, row in df_ucdp.iterrows():
    start, end = row['date_start'], row['date_end']
    months = pd.date_range(start, end, freq='MS')  # one row per month
    n_months = len(months)
    
    tmp = pd.DataFrame({
        'conflict_new_id': row['conflict_new_id'],
        'isocode': row['isocode'],
        'country': row['country'],
        'region': row['region'],
        'dyad_new_id': row['dyad_new_id'],
        'date': months,
        'best': row['best'] / n_months if n_months > 0 else row['best'],
        'type_of_violence': row['type_of_violence'],
        'conflict_name': row['conflict_name'],
        'side_a': row['side_a'],
        'side_b': row['side_b']
    })
    expanded_events.append(tmp)

df_expanded = pd.concat(expanded_events, ignore_index=True)
df_expanded

,conflict_new_id,isocode,country,region,dyad_new_id,date,best,type_of_violence,conflict_name,side_a,side_b
0,259,AFG,Afghanistan,Asia,524,2017-07-01,6.0,1,Iraq: Government,Government of Iraq,IS
1,259,AFG,Afghanistan,Asia,524,2021-08-01,183.0,1,Iraq: Government,Government of Iraq,IS
2,259,AFG,Afghanistan,Asia,524,2021-08-01,2.0,1,Iraq: Government,Government of Iraq,IS
3,259,AFG,Afghanistan,Asia,524,2021-08-01,10.0,1,Iraq: Government,Government of Iraq,IS
4,333,AFG,Afghanistan,Asia,727,1989-01-01,4.0,1,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction
...,...,...,...,...,...,...,...,...,...,...,...
406445,562,ZWE,Zimbabwe (Rhodesia),Africa,1029,1989-03-01,9.0,3,Renamo - Civilians,Renamo,Civilians
406446,562,ZWE,Zimbabwe (Rhodesia),Africa,1029,1989-07-01,9.0,3,Renamo - Civilians,Renamo,Civilians
406447,562,ZWE,Zimbabwe (Rhodesia),Africa,1029,1990-06-01,7.0,3,Renamo - Civilians,Renamo,Civilians
406448,562,ZWE,Zimbabwe (Rhodesia),Africa,1029,1990-11-01,1.0,3,Renamo - Civilians,Renamo,Civilians


In [9]:
#2. Aggregate to conflict-month level and sum the 'best' values without distinguishing countries
#-----------------------------------------------------------------------------------------------
df_conflict_month = (
    df_expanded
    .groupby(['conflict_new_id','date'], as_index=False)
    .agg(
        best=('best','sum'),
        n_events=('dyad_new_id','count'),
        isocode = ('isocode','first'),
        country = ('country','first'),
        region = ('region','first'),
        n_isocode=('isocode','nunique'),
        isocode_array=('isocode', 'unique'),
        countries_array=('country','unique'),
        type_of_violence=('type_of_violence','min')
    )
)
df_conflict_month

,conflict_new_id,date,best,n_events,isocode,country,region,n_isocode,isocode_array,countries_array,type_of_violence
0,205,1990-04-01,0.0,1,IRN,Iran,Middle East,1,[IRN],[Iran],1
1,205,1990-05-01,1.0,1,IRN,Iran,Middle East,1,[IRN],[Iran],1
2,205,1990-06-01,23.0,6,IRN,Iran,Middle East,1,[IRN],[Iran],1
3,205,1990-07-01,7.0,5,IRN,Iran,Middle East,1,[IRN],[Iran],1
4,205,1990-08-01,0.0,1,IRN,Iran,Middle East,1,[IRN],[Iran],1
...,...,...,...,...,...,...,...,...,...,...,...
41549,16527,2024-01-01,36.0,5,MEX,Mexico,Americas,1,[MEX],[Mexico],2
41550,16527,2024-02-01,37.0,16,MEX,Mexico,Americas,1,[MEX],[Mexico],2
41551,16527,2024-03-01,1.0,1,MEX,Mexico,Americas,1,[MEX],[Mexico],2
41552,16527,2024-04-01,1.0,1,MEX,Mexico,Americas,1,[MEX],[Mexico],2


Assign main country to each conflict_id as the country with the highest number of fatalities in that conflict.

In [10]:
# dominant country: the one with most events that month
main_isocode = (
    df_expanded.groupby(['conflict_new_id','isocode', 'country', 'region'])
    .agg(sum_best=('best','sum'))
    .reset_index()
    .sort_values(['conflict_new_id','sum_best'], ascending=[True,False])
    .drop_duplicates(['conflict_new_id'])
)
df_conflict_month = df_conflict_month.merge(
    main_isocode[['conflict_new_id','isocode', 'country', 'region']],
    on=['conflict_new_id'],
    how='left'
)
#drop the columns that should be replaced by main_isocode
df_conflict_month = df_conflict_month.drop(columns=['isocode_x','country_x', 'region_x'])
df_conflict_month = df_conflict_month.rename(columns={'isocode_y':'isocode_main', 'country_y':'country_main', 'region_y':'region_main'})
df_conflict_month

,conflict_new_id,date,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,region_main
0,205,1990-04-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East
1,205,1990-05-01,1.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East
2,205,1990-06-01,23.0,6,1,[IRN],[Iran],1,IRN,Iran,Middle East
3,205,1990-07-01,7.0,5,1,[IRN],[Iran],1,IRN,Iran,Middle East
4,205,1990-08-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East
...,...,...,...,...,...,...,...,...,...,...,...
41549,16527,2024-01-01,36.0,5,1,[MEX],[Mexico],2,MEX,Mexico,Americas
41550,16527,2024-02-01,37.0,16,1,[MEX],[Mexico],2,MEX,Mexico,Americas
41551,16527,2024-03-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,Americas
41552,16527,2024-04-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,Americas


In [11]:
df_conflict_month['best'] = df_conflict_month['best'].fillna(0.0)
df_conflict_month['log_best'] = np.log1p(df_conflict_month['best']) # log(1 + x) to handle 0 values
df_conflict_month = df_conflict_month.rename(columns={'conflict_new_id':'conflict_id', 'date':'year_mo'})
df_conflict_month

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,region_main,log_best
0,205,1990-04-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East,0.000000
1,205,1990-05-01,1.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East,0.693147
2,205,1990-06-01,23.0,6,1,[IRN],[Iran],1,IRN,Iran,Middle East,3.178054
3,205,1990-07-01,7.0,5,1,[IRN],[Iran],1,IRN,Iran,Middle East,2.079442
4,205,1990-08-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,Middle East,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
41549,16527,2024-01-01,36.0,5,1,[MEX],[Mexico],2,MEX,Mexico,Americas,3.610918
41550,16527,2024-02-01,37.0,16,1,[MEX],[Mexico],2,MEX,Mexico,Americas,3.637586
41551,16527,2024-03-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,Americas,0.693147
41552,16527,2024-04-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,Americas,0.693147


### **Agreements**
---


In [12]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')
df_pax.columns = df_pax.columns.str.lower()

df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id", "loc1iso": "isocode"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id', 'year_mo'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

#define variables to test clusterd Heterogenous TE
features_cluster_columns = ['hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis', 
                            'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres', 
                            'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr', 
                            'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
                            'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm', 
                            'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
                            'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme', 
                            'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad', 
                            'juscr', 'mps', 'prot', 'mpspro']

df_agreements = (
    df_pax
    .groupby(["conflict_id", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  
        comp_agreement=("comp_agreement", "max"),  
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max'),
        **{var: (var, "max") for var in features_cluster_columns}
        
    )
)
df_agreements

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/665996309.py:1: DtypeWarning: Columns (23,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')


,conflict_id,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,...,ppsex,hrmob,ele,hrni,civso,pubad,juscr,mps,prot,mpspro
0,209,1992-09-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,209,1994-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,209,1995-02-01,1,0,1,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,209,1995-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,209,1996-06-01,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313,169170,1995-10-01,1,0,1,0,0,0,0,3,...,0,0,1,0,0,0,1,0,1,0
1314,169170,2000-12-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,0,0
1315,169170,2002-08-01,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1316,169170,2002-10-01,1,0,0,0,0,0,0,2,...,0,0,0,0,1,0,0,0,1,0


In [13]:
df_pax = df_pax.drop_duplicates(subset=['conflict_id', 'year_mo'])
relevant_columns = [i for i in df_pax.columns if not i.startswith('g') and i not in ['agreement', 'comp_agreement', 'subs_agreement', 
                                                                                     'cea_agreement', 'cea_ceas_agreement', 
                                                                                     'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns]
df_pax = df_pax[relevant_columns]
df_agreements = df_agreements.merge(
    df_pax,
    on=['conflict_id', 'year_mo'],
    how='left'    
)

In [14]:
df_panel = df_conflict_month.merge(
    df_agreements,
    on=["conflict_id", "year_mo"],
    how="left"
)
df_panel[["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns] = df_panel[
    ["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns
].fillna(0).astype(int)

df_panel['isocode_main_num'] = df_panel['isocode_main'].astype('category').cat.codes + 1
df_panel['region_main_num'] = df_panel['region_main'].astype('category').cat.codes + 1
df_panel['start_date'] = df_panel.groupby('conflict_id')['year_mo'].transform('min')
df_panel['end_date'] = df_panel.groupby('conflict_id')['year_mo'].transform('max')

df_panel

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,tjjaic,tjprire,tjrsym,tjrma,tjnr,imsrc,isocode_main_num,region_main_num,start_date,end_date
0,205,1990-04-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,43,5,1990-04-01,2022-11-01
1,205,1990-05-01,1.0,1,1,[IRN],[Iran],1,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,43,5,1990-04-01,2022-11-01
2,205,1990-06-01,23.0,6,1,[IRN],[Iran],1,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,43,5,1990-04-01,2022-11-01
3,205,1990-07-01,7.0,5,1,[IRN],[Iran],1,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,43,5,1990-04-01,2022-11-01
4,205,1990-08-01,0.0,1,1,[IRN],[Iran],1,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,43,5,1990-04-01,2022-11-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41549,16527,2024-01-01,36.0,5,1,[MEX],[Mexico],2,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,NaN,59,2,2024-01-01,2024-07-01
41550,16527,2024-02-01,37.0,16,1,[MEX],[Mexico],2,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,NaN,59,2,2024-01-01,2024-07-01
41551,16527,2024-03-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,NaN,59,2,2024-01-01,2024-07-01
41552,16527,2024-04-01,1.0,1,1,[MEX],[Mexico],2,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,NaN,59,2,2024-01-01,2024-07-01


## **Expand the panel**

From 1989-01 to 2024-12 monthly, but keeping the start_date and end_date of each conflict as in the original UCDP conflict dataset, to calculate conflict age and duration.


In [15]:
#4. Fill missing months with 0 fatalities to expand to a balanced panel
#-----------------------------------------------------------------------------------------------

global_start = df_ucdp.date_start.min()
global_end   = df_ucdp.date_end.max()

full_conflict_month = []

for conflict_id in df_panel['conflict_id'].unique():
    months = pd.date_range(global_start, global_end, freq='MS')
    tmp = pd.DataFrame({'conflict_id': conflict_id, 'year_mo': months})
    full_conflict_month.append(tmp)

full_panel = pd.concat(full_conflict_month, ignore_index=True)

# merge to fill missing months with 0 fatalities
df_panel = full_panel.merge(df_panel, on=['conflict_id','year_mo'], how='left')
df_panel = df_panel.sort_values(['conflict_id', 'year_mo']).reset_index(drop=True)
df_panel['real_observation'] = np.where(df_panel['best'].notna(), 1, 0)


# Fill in missing values
columns_to_fill = ['best', 'agreement', 'comp_agreement','subs_agreement', 
                   'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement',
                   'cea_rel_agreement','n_events','n_isocode', 'ce'] + features_cluster_columns
for i in columns_to_fill:
    df_panel[i] = df_panel.groupby('conflict_id')[i].fillna(0)

df_panel['log_best'] = np.log1p(df_panel['best'])

columns_to_ffill_bfill = ['isocode_main', 'country_main', 'isocode_main_num', 'region_main','region_main_num', 'isocode_array', 'countries_array', 'type_of_violence', 'start_date', 'end_date']
for i in columns_to_ffill_bfill:
    df_panel[i] = df_panel.groupby('conflict_id')[i].transform(lambda x: x.ffill().bfill())

df_panel

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/490473579.py:27: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_panel[i] = df_panel.groupby('conflict_id')[i].fillna(0)


,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,tjprire,tjrsym,tjrma,tjnr,imsrc,isocode_main_num,region_main_num,start_date,end_date,real_observation
0,205,1989-01-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,43.0,5.0,1990-04-01,2022-11-01,0
1,205,1989-02-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,43.0,5.0,1990-04-01,2022-11-01,0
2,205,1989-03-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,43.0,5.0,1990-04-01,2022-11-01,0
3,205,1989-04-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,43.0,5.0,1990-04-01,2022-11-01,0
4,205,1989-05-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,43.0,5.0,1990-04-01,2022-11-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661387,16527,2024-08-01,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,59.0,2.0,2024-01-01,2024-07-01,0
661388,16527,2024-09-01,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,59.0,2.0,2024-01-01,2024-07-01,0
661389,16527,2024-10-01,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,59.0,2.0,2024-01-01,2024-07-01,0
661390,16527,2024-11-01,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,NaN,NaN,NaN,NaN,NaN,59.0,2.0,2024-01-01,2024-07-01,0


## **Define Features**
---
- conflict_age: the age of the conflict in months as the number of months between start_date and the current month
- active_conflict_age: the age of the conflict considering only months with fatalities reported
- duration_months: the number of months between start_date and end_date of each conflict (max end_date - min start_date)
- active_duration_months: the number of real observations (months with fatalities reported)


In [16]:
df_panel['year_mo'] = pd.to_datetime(df_panel['year_mo'], format='%Y-%m').dt.to_period('M')
df_panel['start_date'] = pd.to_datetime(df_panel['start_date'], format='%Y-%m').dt.to_period('M')
df_panel['end_date'] = pd.to_datetime(df_panel['end_date'], format='%Y-%m').dt.to_period('M')

base_date = df_panel['year_mo'].min()

#numeric month index starting from 1
df_panel['year_mo_numeric'] = (df_panel["year_mo"] - base_date).apply(lambda x: x.n+1)
df_panel['start_date_numeric'] = (df_panel['start_date'] - base_date).apply(lambda x: x.n+1)
df_panel['end_date_numeric'] = (df_panel['end_date'] - base_date).apply(lambda x: x.n+1)

#Conflict Age
#-----------------------------------------------------------------------------------------------
df_panel["conflict_age"] = df_panel['year_mo_numeric']- df_panel['start_date_numeric']
#for the negative values in all the cases impute with missing
df_panel['conflict_age'] = df_panel['conflict_age'].apply(lambda x: x if x >=0 else np.nan)

df_panel['conflict_age_less_6m']  = np.where(df_panel['conflict_age'].notna(), (df_panel['conflict_age'] <= 6).astype(int), np.nan)
df_panel['conflict_age_less_12m'] = np.where(df_panel['conflict_age'].notna(), (df_panel['conflict_age'] <= 12).astype(int), np.nan)
df_panel['conflict_age_less_18m'] = np.where(df_panel['conflict_age'].notna(), (df_panel['conflict_age'] <= 18).astype(int), np.nan)
df_panel['conflict_age_less_24m'] = np.where(df_panel['conflict_age'].notna(), (df_panel['conflict_age'] <= 24).astype(int), np.nan)
df_panel['conflict_age_less_30m'] = np.where(df_panel['conflict_age'].notna(), (df_panel['conflict_age'] <= 30).astype(int), np.nan)


#Active Conflict Age
#-----------------------------------------------------------------------------------------------
df_panel["active_conflict_age"] = (
    df_panel.groupby("conflict_id").apply(
        lambda group: group["real_observation"].cumsum() - 1
    ).reset_index(level=0, drop=True)
)
df_panel['active_conflict_age'] = df_panel['active_conflict_age'].apply(lambda x: x if x >=0 else np.nan)

#Duration Conflict in Months
#-----------------------------------------------------------------------------------------------
#difference between end_date_numeric and start_date_numeric
df_panel['duration_months'] = (df_panel.groupby("conflict_id")["end_date_numeric"].transform("max") - df_panel.groupby("conflict_id")["start_date_numeric"].transform("min"))

# #Duration of Active Conflict in Months
# #-----------------------------------------------------------------------------------------------
df_panel["active_duration_months"] = (
     df_panel.groupby("conflict_id")["real_observation"].transform(lambda x: x.sum())
     )


# start_year from start_date
df_panel['start_year'] = df_panel['start_date'].dt.year

# current month from year_mo
df_panel['current_month'] = df_panel['year_mo'].dt.month

df_panel


/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/1192411326.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_panel.groupby("conflict_id").apply(
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/1192411326.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_panel["active_duration_months"] = (
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/1192411326.py:47: PerformanceWarning: DataFrame is highly fragmented.  This 

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,conflict_age_less_6m,conflict_age_less_12m,conflict_age_less_18m,conflict_age_less_24m,conflict_age_less_30m,active_conflict_age,duration_months,active_duration_months,start_year,current_month
0,205,1989-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,1
1,205,1989-02,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,2
2,205,1989-03,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,3
3,205,1989-04,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,4
4,205,1989-05,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,NaN,NaN,NaN,NaN,NaN,NaN,391,47,1990,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661387,16527,2024-08,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0.0,1.0,1.0,1.0,1.0,4.0,6,5,2024,8
661388,16527,2024-09,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0.0,1.0,1.0,1.0,1.0,4.0,6,5,2024,9
661389,16527,2024-10,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0.0,1.0,1.0,1.0,1.0,4.0,6,5,2024,10
661390,16527,2024-11,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0.0,1.0,1.0,1.0,1.0,4.0,6,5,2024,11


### **First treatment date per conflict**
---




In [17]:
for i in ['agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement']:
    #create the group variable for first treatment date
    df_panel[f'first_{i}_date'] = (
    df_panel[df_panel[i] == 1]
    .groupby('conflict_id')['year_mo_numeric']
    .transform('min')
    )

    df_panel[f'first_{i}_date'] = (
    df_panel.groupby('conflict_id')[f'first_{i}_date']
    .transform(lambda x: x.ffill().bfill())
    )
    #Variable gvar: 0 = never treated, positive = month of the first treatment
    df_panel[f'first_{i}_date'] = df_panel[f'first_{i}_date'].fillna(0).astype(int)

    #Dummy: indicator for treated units by agreements
    df_panel[f'treated_{i}'] = (
    df_panel.groupby('conflict_id')[i]
    .transform('max')
    )

df_panel

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/2036749001.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_panel[f'first_{i}_date'] = (
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/2036749001.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_panel[f'treated_{i}'] = (
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_83705/2036749001.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor p

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,205,1989-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
1,205,1989-02,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,205,1989-03,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
3,205,1989-04,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
4,205,1989-05,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661387,16527,2024-08,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
661388,16527,2024-09,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
661389,16527,2024-10,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
661390,16527,2024-11,0.0,0.0,0.0,[MEX],[Mexico],2.0,MEX,Mexico,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [18]:
df_panel = df_panel.loc[df_panel['type_of_violence'] == 1].reset_index(drop=True).copy()
df_panel

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,205,1989-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
1,205,1989-02,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,205,1989-03,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
3,205,1989-04,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
4,205,1989-05,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
86828,16379,2024-09,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
86829,16379,2024-10,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
86830,16379,2024-11,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [19]:
def gen_since(series: pd.Series):
    series = (series == False)
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()
    result[groups == 0] = np.nan
    return result

df_panel['since_last_agreement'] = df_panel.groupby('conflict_id')['agreement'].transform(gen_since)


In [20]:
def gen_sequence_since_start(since_last_agreement, year_mo_numeric, first_agreement_date, sequence_size=12):
    """
    Generates cumulative sequence numbers that start counting
    only after the conflict's start_date.

    Parameters
    ----------
    since_last_agreement : pd.Series
        Series indicating months since last agreement.
    year_mo_numeric : pd.Series
        Numeric month-year index (1 for 1989-01, etc.).
    start_date : float or int
        Numeric start date of the conflict.
    sequence_size : int
        Threshold (in months) for incrementing sequence count.

    Returns
    -------
    pd.Series
        sequence indicator starting at 1 from the start_date.
        NaN before the conflict starts, and 0 if never treated.
    """

    # Create a copy to avoid modifying the original
    s = since_last_agreement.copy()
    seq = pd.Series(index=s.index, dtype="float")

    # Mask: only months after conflict start
    active_mask = year_mo_numeric >= first_agreement_date - sequence_size

    # Only compute sequence for months where conflict is active
    s_active = s[active_mask]

    if s_active.notna().sum() == 0:
        # No valid agreements after start_date → return NaN (filled with 0 later)
        seq.loc[:] = np.nan
        return seq

    # Detect when since_last_agreement crosses the threshold
    trigger = (s_active.shift(1) <= sequence_size) & (s_active == sequence_size + 1)

    # Cumulative sum + 1 = sequence number
    seq_active = trigger.cumsum() + 1
    seq_active = seq_active.fillna(1).astype(int)

    # Assign results
    seq.loc[active_mask] = seq_active

    # Before start_date, keep NaN (will be filled later with 0)
    return seq


In [21]:
df_panel = df_panel.sort_values(['conflict_id', 'year_mo_numeric'])

#Sequence for 12 months
#------------------------------------------------------------------------
df_panel['sequence_12m'] = np.nan  # initialize

treated = df_panel['treated_agreement'] == 1

for cid, group in df_panel.loc[treated].groupby('conflict_id'):
    start = group['first_agreement_date'].iloc[0]
    seq = gen_sequence_since_start(
        group['since_last_agreement'],
        group['year_mo_numeric'],
        start,
        sequence_size=12
    )
    df_panel.loc[group.index, 'sequence_12m'] = seq

# Fill 0 for untreated conflicts
df_panel.loc[~treated, 'sequence_12m'] = 0
df_panel['sequence_12m'] = df_panel['sequence_12m'].astype('Int64')  

In [22]:
df_panel.columns = df_panel.columns.str.replace(' ', '_').str.replace('-', '_').str.replace('.', '_').str.replace('|', '_')
df_panel['conflict_id'] = df_panel['conflict_id'].astype(int)
df_panel['best'] = df_panel['best'].astype(float)
df_panel['log_best'] = df_panel['log_best'].astype(float)
#df_panel = pd.get_dummies(df_panel, columns=['stage', 'stagesub', 'agtp', 'status'], prefix=['stage', 'stagesub', 'agtp', 'status'])

df_panel

,conflict_id,year_mo,best,n_events,n_isocode,isocode_array,countries_array,type_of_violence,isocode_main,country_main,...,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement,since_last_agreement,sequence_12m
0,205,1989-01,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
1,205,1989-02,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
2,205,1989-03,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
3,205,1989-04,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
4,205,1989-05,0.0,0.0,0.0,[IRN],[Iran],1.0,IRN,Iran,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86827,16379,2024-08,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
86828,16379,2024-09,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
86829,16379,2024-10,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0
86830,16379,2024-11,0.0,0.0,0.0,[SOM],[Somalia],1.0,SOM,Somalia,...,0,0.0,0,0.0,0,0.0,0,0.0,NaN,0


In [23]:
df_panel.to_csv('../../data/output/conflict_level/conflict_panel.csv')